# LGM 1F Demo + Performance Counter

This notebook demonstrates the extracted `py_ore_tools.lgm` module and benchmarks simulation speed using `time.perf_counter()`.

In [ ]:
import time
import numpy as np

from py_ore_tools.lgm import LGMParams, LGM1F, simulate_lgm_measure, simulate_ba_measure

In [ ]:
# Model setup (Hagan vol + HW reversion, piecewise constants)
params = LGMParams(
    alpha_times=(1.0, 3.0, 7.0),
    alpha_values=(0.015, 0.020, 0.018, 0.012),
    kappa_times=(2.0, 5.0),
    kappa_values=(0.030, 0.025, 0.020),
    shift=0.0,
    scaling=1.0,
)
model = LGM1F(params)

# Time grid for exposure/simulation style runs
times = np.linspace(0.0, 30.0, 121)  # 120 steps
times[:5], times[-5:]

In [ ]:
def benchmark(name, func, repeats=5, warmup=1):
    for _ in range(warmup):
        _ = func()

    durations = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = func()
        t1 = time.perf_counter()
        durations.append(t1 - t0)

    arr = np.array(durations, dtype=float)
    print(f"{name}:")
    print(f"  runs={repeats}, mean={arr.mean():.6f}s, std={arr.std(ddof=0):.6f}s, min={arr.min():.6f}s")
    return out, arr


In [ ]:
def run_lgm(n_paths):
    rng = np.random.default_rng(42)
    return simulate_lgm_measure(model, times, n_paths=n_paths, rng=rng, x0=0.0)

def run_ba(n_paths):
    rng = np.random.default_rng(42)
    return simulate_ba_measure(model, times, n_paths=n_paths, rng=rng, x0=0.0, y0=0.0)

for n in [2000, 10000, 50000]:
    paths_steps = n * (len(times) - 1)

    _, lgm_t = benchmark(
        f"LGM measure (n_paths={n})",
        lambda n=n: run_lgm(n),
        repeats=5,
        warmup=1,
    )
    print(f"  throughput ~ {paths_steps / lgm_t.mean():,.0f} path-steps/s")

    _, ba_t = benchmark(
        f"BA measure (n_paths={n})",
        lambda n=n: run_ba(n),
        repeats=5,
        warmup=1,
    )
    print(f"  throughput ~ {paths_steps / ba_t.mean():,.0f} path-steps/s")
    print()

In [ ]:
# Quick sanity checks
x = run_lgm(20000)
xT = x[-1]
print("E[x(T)] ~", xT.mean())
print("Var[x(T)] ~", xT.var())
print("zeta(T)   =", float(model.zeta(times[-1])))